# generar tripletas para 2 cohortes

In [ ]:
import pandas as pd
import os
import random

# ================================
# CONFIGURACIÓN
# ================================
BASE_CSV = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"

# Pares (semestre_anterior, semestre_actual) que quieres mezclar
PARES_SEMESTRES = [
    ("20191", "20192"),  # generación 2019
    ("20201", "20202"),  # generación 2020
]

# Cursos (asumo misma malla 2019–2020; cámbialos si difieren)
cursos_primer_semestre = {"MA1101", "MA1001", "FI1000", "BT1211"}
cursos_segundo_semestre = {"CC1002", "MA1002", "MA1102", "FI1100"}
cursos_permitidos = cursos_primer_semestre.union(cursos_segundo_semestre)

# Carpeta donde se guardará el dataset combinado
output_dir = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales"

# Fijar semilla para reproducibilidad
random.seed(42)

# ================================
# FUNCIÓN AUXILIAR
# ================================

def normalizar_columnas(df):
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    df["CURSO"] = df["CURSO"].astype(str).str.strip().str.upper()
    df["ESTADO_CURSO"] = df["ESTADO_CURSO"].astype(str)
    return df

# ================================
# 1) RECORRER TODAS LAS GENERACIONES
# ================================

tripletas = []

for semestre_anterior, semestre_actual in PARES_SEMESTRES:
    ruta_prev = os.path.join(BASE_CSV, f"df_{semestre_anterior}.csv")
    ruta_act  = os.path.join(BASE_CSV, f"df_{semestre_actual}.csv")

    print(f"\n=== Procesando cohorte {semestre_anterior} → {semestre_actual} ===")
    print(f"CSV previo : {ruta_prev}")
    print(f"CSV actual : {ruta_act}")

    df_prev = pd.read_csv(ruta_prev, sep=";")
    df_act  = pd.read_csv(ruta_act,  sep=";")

    df_prev = normalizar_columnas(df_prev)
    df_act  = normalizar_columnas(df_act)

    # --- 2) Alumnos con todos los cursos fundamentales del 1er semestre ---
    df_fund_prev = df_prev[df_prev["CURSO"].isin(cursos_primer_semestre)]
    conteo = df_fund_prev.groupby("ID")["CURSO"].nunique()
    alumnos_validos = conteo[conteo == len(cursos_primer_semestre)].index

    print(f"✅ Alumnos con los 4 cursos de 1er semestre en {semestre_anterior}: {len(alumnos_validos)}")

    # --- 3) Inscripciones de esos alumnos en el semestre actual, solo cursos permitidos ---
    df_filtrado = df_act[(df_act["ID"].isin(alumnos_validos)) &
                         (df_act["CURSO"].isin(cursos_permitidos))]

    # --- 4) Crear tripletas para esta cohorte ---
    for _, row in df_filtrado.iterrows():
        alumno = row["ID"]
        curso  = row["CURSO"]
        estado = row["ESTADO_CURSO"]

        if "Aprobado" in estado:
            rel = "aprueba"
        elif ("Reprobado" in estado) or ("Eliminado" in estado):
            # tratamos Eliminado como reprueba, igual que Reprobado
            rel = "reprueba"
        else:
            # estados como "Convalidado", "Con retiro", etc. se omiten
            continue

        tripletas.append((alumno, rel, curso))

    print(f"📊 Tripletas acumuladas hasta ahora: {len(tripletas)}")

# ================================
# 5) DIVIDIR EN TRAIN / VALID / TEST
# ================================

print(f"\n📊 Total de tripletas generadas (2019 + 2020): {len(tripletas)}")

random.shuffle(tripletas)
n_total = len(tripletas)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_triplets = tripletas[:n_train]
valid_triplets = tripletas[n_train:n_train + n_valid]
test_triplets  = tripletas[n_train + n_valid:]

# ================================
# 6) GUARDAR EN FORMATO TuckER
# ================================

os.makedirs(output_dir, exist_ok=True)

def guardar_tripletas(nombre, data):
    ruta = os.path.join(output_dir, nombre)
    with open(ruta, "w", encoding="utf-8") as f:
        for h, r, t in data:
            f.write(f"{h}\t{r}\t{t}\n")
    print(f"  -> {nombre}: {len(data)} tripletas")

print(f"\n💾 Guardando archivos en {output_dir}")
guardar_tripletas("train.txt", train_triplets)
guardar_tripletas("valid.txt", valid_triplets)
guardar_tripletas("test.txt",  test_triplets)

print("\n✅ Listo.")
print(f"Train: {len(train_triplets)} | Valid: {len(valid_triplets)} | Test: {len(test_triplets)}")


# Probar modelo con dos cohortes

In [1]:
# -*- coding: utf-8 -*-
import os, re, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from types import SimpleNamespace
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# =====================================
# RUTAS / PARÁMETROS (comunes)
# =====================================
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales\\"

# CSVs con notas
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20131 = os.path.join(BASE_PATH, "df_20211.csv")  # features
CSV_20132 = os.path.join(BASE_PATH, "df_20212.csv")  # etiquetas

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']        # entrada predictor
CURSOS_SEGUNDO = ['CC1002','MA1002','MA1102','FI1100']        # eval típica
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

DEVICE = "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# =====================================
# PATRONES DE RUTAS (ajusta si cambian)
# =====================================
RESULTS_BASE = r"C:\Users\56946\TuckER\results"
RUN_PREFIX   = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience300"
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start"

# Recorrer rdim = 1..16 (sin excepciones)
RDIMS = list(range(1, 17))

# =========================
# UTILIDADES
# =========================
sys.path.append("C:/Users/56946/TuckER")
from load_data import Data  # para leer vocab (no se entrena)

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    ent2idx = {e:i for i,e in enumerate(d.entities)}
    rel2idx = {r:i for i,r in enumerate(d.relations)}
    return SimpleNamespace(entities=d.entities, relations=d.relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt and isinstance(ckpt["model_state_dict"], dict):
            return ckpt["model_state_dict"]
        if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
            return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(path, device="cpu"):
    sd = pick_state_dict(torch.load(path, map_location=device))
    E = sd["E.weight"].to(device)  # [|E|, d_e]
    R = sd["R.weight"].to(device)  # [|R|, d_r]
    W = sd["W"].to(device)         # [d_r, d_e, d_e] o [d_e, d_r, d_e]
    return E, R, W

def pick_forward_rel_idx(vocab, name):
    if name in vocab.relation_idxs:
        return vocab.relation_idxs[name]
    cands = [r for r in vocab.relations if r.replace("_reverse","") == name]
    if not cands:
        cands = [r for r in vocab.relations if re.search(name, r, re.I)]
    if not cands:
        raise KeyError(f"No encontré relación '{name}' en el vocab.")
    cands.sort(key=lambda r: ("_reverse" in r, r))
    return vocab.relation_idxs[cands[0]]

def contract_M(W, r_vec, d_e):
    # Contrae el core W con el vector de relación r -> M_r en [d_e, d_e]
    if W.shape[0] == r_vec.numel() and W.shape[1] == d_e:
        return torch.tensordot(W, r_vec, dims=([0],[0]))
    if W.shape[1] == r_vec.numel() and W.shape[0] == d_e:
        return torch.tensordot(W, r_vec, dims=([1],[0]))
    raise ValueError(f"Layout W no reconocido: {tuple(W.shape)} vs d_e={d_e}, d_r={r_vec.numel()}")

@torch.no_grad()
def prob_relacion_para_par(e_h, e_t, R, W, ridx):
    d_e = e_h.shape[0]
    M_r = contract_M(W, R[ridx], d_e)       # [d_e, d_e]
    s   = (e_h.view(1,d_e) @ M_r @ e_t.view(d_e,1)).squeeze()
    return torch.sigmoid(s).item()

# ===== Predictor (notas -> embedding) =====
class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def load_predictor(path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=device)
    if isinstance(state, dict) and any(k in state for k in ("model_state_dict","state_dict")):
        state = state.get("model_state_dict", state.get("state_dict"))
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model

def notas_vector(csv_path, alumno_id, cursos_primer):
    df = pd.read_csv(csv_path, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    sub = df[(df['ID']==alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    if not sub.empty and sub['NOTA'].isna().all():
        sub['NOTA'] = 0.0
    if sub.empty:
        return torch.zeros((1,len(cursos_primer)), dtype=torch.float32)
    piv = (sub.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
              .reindex(columns=cursos_primer).fillna(0.0))
    vec = piv.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1,-1)

# =========================
# EVALUACIÓN (una corrida)
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab):
    print(f"\n=============== Evaluando modelo: {tag} ===============")
    # Carga pesos TuckER
    E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
    d_e = E.shape[1]

    # Índices de relaciones
    try:
        ridx_apr  = pick_forward_rel_idx(vocab, "aprueba")
    except KeyError:
        ridx_apr  = pick_forward_rel_idx(vocab, "aprob")
    try:
        ridx_repr = pick_forward_rel_idx(vocab, "reprueba")
        have_repr = True
    except KeyError:
        have_repr = False
        print("⚠️ No hay relación 'reprueba' en vocab. Usaré 1 - P(aprueba).")

    # Predictor
    predictor = load_predictor(predictor_ckpt, input_size=len(CURSOS_PRIMER), out_dim=d_e, device=DEVICE)

    # Datos 20131 (features) y 20132 (etiquetas y cursos)
    df_31 = pd.read_csv(CSV_20131, sep=';')
    df_32 = pd.read_csv(CSV_20132, sep=';')
    for df in (df_31, df_32):
        df['ID']    = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # Alumnos con los 4 cursos del 1er semestre en 20131
    ok = (df_31.groupby("ID")["CURSO"].apply(set)
             .apply(lambda s: set(CURSOS_PRIMER).issubset(s)))
    alumnos_validos = ok[ok].index.tolist()

    # Filas a evaluar
    df_eval = df_32[(df_32['ID'].isin(alumnos_validos)) &
                    (df_32['CURSO'].isin(CURSOS_EVAL))].copy()

    # etiqueta binaria
    df_eval['APROB'] = ((~df_eval['NOTA'].isna()) & (df_eval['NOTA'] >= 4.0)).astype(int)

    # filtrar cursos presentes en el vocab
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())].copy()
    if df_eval.empty:
        print("⚠️ No hay pares alumno-curso evaluables (faltan entidades en vocab).")
        return None, None

    # cache y evaluación
    ehat_cache = {}
    rows = []
    
    # Listas para métricas de clasificación (Clase Positiva = Reprueba)
    y_true_cls = []
    y_pred_cls = []

    with torch.no_grad():
        M_apr = contract_M(W, R[ridx_apr], d_e)
        M_repr = contract_M(W, R[ridx_repr], d_e) if have_repr else None

        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            if aid not in ehat_cache:
                x = notas_vector(CSV_20131, aid, CURSOS_PRIMER).to(DEVICE)
                ehat_cache[aid] = predictor(x).squeeze(0).cpu()
            e_h = ehat_cache[aid]
            e_t = E[vocab.entity_idxs[curso]].cpu()

            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            if have_repr:
                s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item()
            else:
                s_repr = 1.0 - s_apr

            # --- Lógica de Ranking (Hits/MRR) ---
            if r['APROB'] == 1:
                true = 'aprueba'
                rank = 1 if s_apr >= s_repr else 2
            else:
                true = 'reprueba'
                rank = 1 if s_repr >= s_apr else 2

            # --- Lógica de Clasificación (Reprobado) ---
            # Definimos: Clase 1 = Reprobado, Clase 0 = Aprobado
            es_reprobado_real = 1 if r['APROB'] == 0 else 0
            predice_reprobado = 1 if s_repr > s_apr else 0
            
            y_true_cls.append(es_reprobado_real)
            y_pred_cls.append(predice_reprobado)

            rows.append({
                "ID": aid, "CURSO": curso, "APROB": int(r['APROB']),
                "P_aprueba": s_apr, "P_reprueba": s_repr,
                "true_rel": true, "rank_rel": rank,
                "hit@1": 1.0 if rank == 1 else 0.0,
                "hit@2": 1.0
            })

    df_out = pd.DataFrame(rows)
    ranks = df_out["rank_rel"].to_numpy(dtype=float)
    hits1 = df_out["hit@1"].mean()
    hits2 = df_out["hit@2"].mean()
    mr    = ranks.mean()
    mrr   = (1.0 / ranks).mean()
    N     = len(df_out)

    # --- Cálculo de Precision y Recall para Reprobados ---
    prec_repro = precision_score(y_true_cls, y_pred_cls, zero_division=0)
    rec_repro  = recall_score(y_true_cls, y_pred_cls, zero_division=0)
    
    # Matriz de Confusión: TN(Apr-Apr), FP(Apr-Rep), FN(Rep-Apr), TP(Rep-Rep)
    tn, fp, fn, tp = confusion_matrix(y_true_cls, y_pred_cls).ravel()

    print("\n=== MÉTRICAS GENERALES ===")
    print(f"Hits@1 : {hits1:.4f}")
    print(f"MRR    : {mrr:.4f}")
    print(f"(N={N})")
    
    print("\n=== MÉTRICAS CLASE REPROBADOS ===")
    print(f"Precision (Reprobado): {prec_repro:.4f}")
    print(f"Recall (Reprobado)   : {rec_repro:.4f}")
    print(f"Confusion Matrix     : [TN={tn} (Ok), FP={fp}]")
    print(f"                       [FN={fn} (Miss), TP={tp} (Hit)]")

    # Guardar detalle
    out_csv = os.path.join(BASE_PATH, f"detalle_relrank_aprueba_reprueba_20212_{tag}_ultimo.csv")
    df_out.to_csv(out_csv, index=False)
    print(f"💾 Detalle guardado en: {out_csv}")

    resumen = {
        "tag": tag,
        "Hits@1": hits1,
        "Hits@2": hits2,
        "MR": mr,
        "MRR": mrr,
        "Precision_Rep": prec_repro,
        "Recall_Rep": rec_repro,
        "N": N
    }
    return df_out, resumen

def main():
    # Vocab común
    vocab = build_vocab(DATA_DIR, reverse=True)

    res_list = []
    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        # Rutas auto-generadas para cada rdim
        tucker_ckpt = os.path.join(
            RESULTS_BASE,
            RUN_PREFIX.format(rdim=rdim),
            "best_model.pt"
        )
        predictor_ckpt = os.path.join(
            PREDICTOR_BASE,
            f"best_predictor_model_rdim{rdim}_normalizado_2019_2020.pt"
        )

        # Chequeos
        if not os.path.exists(tucker_ckpt):
            print(f"❌ No existe: {tucker_ckpt}")
            continue
        if not os.path.exists(predictor_ckpt):
            print(f"❌ No existe: {predictor_ckpt}")
            continue

        _, resumen = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab)
        if resumen is not None:
            res_list.append(resumen)

    if res_list:
        df_sum = pd.DataFrame(res_list).sort_values("tag")
        out_sum = os.path.join(BASE_PATH, "resumen_relrank_por_modelo_20212_20192020sv_con_metricas_repro.csv")
        df_sum.to_csv(out_sum, index=False)
        print("\n================ RESUMEN MODELOS ================")
        print(df_sum.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        print(f"\n💾 Resumen guardado en: {out_sum}")
    else:
        print("⚠️ No se generaron resultados. Revisa las rutas de checkpoints/predictors.")

if __name__ == "__main__":
    main()


=============== Evaluando modelo: rdim1 ===============

=== MÉTRICAS GENERALES ===
Hits@1 : 0.0999
MRR    : 0.5500
(N=3023)

=== MÉTRICAS CLASE REPROBADOS ===
Precision (Reprobado): 0.0650
Recall (Reprobado)   : 0.7983
Confusion Matrix     : [TN=116 (Ok), FP=2674]
                       [FN=47 (Miss), TP=186 (Hit)]
💾 Detalle guardado en: C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\detalle_relrank_aprueba_reprueba_20212_rdim1_ultimo.csv

=============== Evaluando modelo: rdim2 ===============

=== MÉTRICAS GENERALES ===
Hits@1 : 0.9001
MRR    : 0.9500
(N=3023)

=== MÉTRICAS CLASE REPROBADOS ===
Precision (Reprobado): 0.3349
Recall (Reprobado)   : 0.3004
Confusion Matrix     : [TN=2651 (Ok), FP=139]
                       [FN=163 (Miss), TP=70 (Hit)]
💾 Detalle guardado en: C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\detalle_relrank_aprueba_reprueba_20212_rdim2_ultimo.csv

=============== Evaluando modelo: rdim3 ===============

=== MÉTRICAS GENERALES ===
Hit

### variando el threshold combinando reprobados y aprobados con [4,5)

In [2]:
# -*- coding: utf-8 -*-
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from types import SimpleNamespace

# =====================================
# CONFIGURACIÓN
# =====================================

# ⚠️ Seleccionamos 4 modelos con >90% Acc Global en la evaluación previa
# Basado en tu output: 6, 10, 13 son muy buenos. Agregamos 16.
RDIMS_SELECCIONADOS = [6, 10, 13, 16]

# Rutas Base
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RESULTS_BASE = r"C:\Users\56946\TuckER\results"
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales\\"
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start"

# Patrones de Archivos
RUN_PREFIX   = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience300"
PREDICTOR_FILE = "best_predictor_model_rdim{rdim}_normalizado_2019_2020.pt"

# Datos
CSV_20131 = os.path.join(BASE_PATH, "df_20211.csv") # Input
CSV_20132 = os.path.join(BASE_PATH, "df_20212.csv") # Target

CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['CC1002','MA1002','MA1102','FI1100']
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

DEVICE = "cpu"

# =====================================
# UTILIDADES
# =====================================
sys.path.append("C:/Users/56946/TuckER")
from load_data import Data 

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    ent2idx = {e:i for i,e in enumerate(d.entities)}
    rel2idx = {r:i for i,r in enumerate(d.relations)}
    return SimpleNamespace(entities=d.entities, relations=d.relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(path, device="cpu"):
    sd = pick_state_dict(torch.load(path, map_location=device))
    return sd["E.weight"].to(device), sd["R.weight"].to(device), sd["W"].to(device)

def pick_forward_rel_idx(vocab, name):
    cands = [r for r in vocab.relations if r.replace("_reverse","") == name]
    if not cands: cands = [r for r in vocab.relations if name in r]
    if not cands: raise KeyError(f"No encontré relación '{name}'")
    cands.sort(key=lambda r: ("_reverse" in r, r))
    return vocab.relation_idxs[cands[0]]

def contract_M(W, r_vec, d_e):
    if W.shape[0] == r_vec.numel() and W.shape[1] == d_e:
        return torch.tensordot(W, r_vec, dims=([0],[0]))
    if W.shape[1] == r_vec.numel() and W.shape[0] == d_e:
        return torch.tensordot(W, r_vec, dims=([1],[0]))
    return torch.tensordot(W, r_vec, dims=([0],[0])) # Fallback

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    sd = torch.load(path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(sd))
    model.to(DEVICE).eval()
    return model

def notas_vector(csv_path, alumno_id, cursos_primer):
    try: df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, len(cursos_primer)))
    
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    
    sub = df[(df['ID']==alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    if sub.empty: return torch.zeros((1,len(cursos_primer)))
    
    sub['NOTA'] = pd.to_numeric(sub['NOTA'], errors='coerce').fillna(0.0)
    piv = sub.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first") \
             .reindex(columns=cursos_primer).fillna(0.0)
    vec = piv.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1,-1)

# =====================================
# BARRIDO DE UMBRALES
# =====================================
def analizar_umbral_riesgo(rdim, vocab, umbrales):
    print(f"\n⚡ Analizando RDIM={rdim}...")
    
    # Rutas
    tucker_path = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
    pred_path = os.path.join(PREDICTOR_BASE, PREDICTOR_FILE.format(rdim=rdim))
    
    if not os.path.exists(tucker_path) or not os.path.exists(pred_path):
        print(f"❌ Faltan archivos para rdim={rdim}")
        return []

    # Cargar Modelos
    E, R, W = load_tucker_weights(tucker_path, DEVICE)
    d_e = E.shape[1]
    
    try:
        ridx_apr = pick_forward_rel_idx(vocab, "aprueba")
        ridx_rep = pick_forward_rel_idx(vocab, "reprueba")
    except:
        print("❌ Error en relaciones del vocabulario")
        return []

    predictor = load_predictor(pred_path, len(CURSOS_PRIMER), d_e)

    # Cargar Datos
    df_31 = pd.read_csv(CSV_20131, sep=';')
    df_32 = pd.read_csv(CSV_20132, sep=';')
    for df in (df_31, df_32):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # Filtrar Alumnos Válidos
    ok = (df_31.groupby("ID")["CURSO"].apply(set).apply(lambda s: set(CURSOS_PRIMER).issubset(s)))
    validos = ok[ok].index.tolist()
    
    df_eval = df_32[(df_32['ID'].isin(validos)) & (df_32['CURSO'].isin(CURSOS_EVAL))].copy()
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())]
    
    # Pre-calcular Scores del Modelo (Solo una vez)
    # El modelo predice "Reprueba" (Clase 1) si P(reprueba) > P(aprueba)
    y_pred_model = []
    notas_reales = []
    estados_reales = []
    
    ehat_cache = {}
    
    with torch.no_grad():
        M_apr = contract_M(W, R[ridx_apr], d_e)
        M_rep = contract_M(W, R[ridx_rep], d_e)
        
        for _, row in df_eval.iterrows():
            aid, curso = row['ID'], row['CURSO']
            
            if aid not in ehat_cache:
                x = notas_vector(CSV_20131, aid, CURSOS_PRIMER).to(DEVICE)
                ehat_cache[aid] = predictor(x).squeeze(0)
            
            e_h = ehat_cache[aid]
            e_t = E[vocab.entity_idxs[curso]]
            
            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            s_rep = torch.sigmoid((e_h.view(1,d_e) @ M_rep @ e_t.view(d_e,1)).squeeze()).item()
            
            # Predicción del modelo: 1 si predice REPRUEBA, 0 si APRUEBA
            pred = 1 if s_rep > s_apr else 0
            y_pred_model.append(pred)
            
            # Guardar datos reales para el barrido
            nota_val = 0.0
            try: nota_val = float(str(row['NOTA']).replace(",", "."))
            except: pass
            notas_reales.append(nota_val)
            estados_reales.append(str(row['ESTADO_CURSO']))

    # --- BARRIDO DE UMBRALES ---
    resultados_rdim = []
    
    print(f"   Evaluando {len(y_pred_model)} registros contra umbrales variables...")
    
    for umbral in umbrales:
        # Definir Ground Truth VARIABLE según el umbral
        # Clase 1 (Riesgo) = Reprobado OR Nota < Umbral
        # Clase 0 (Ok)     = Aprobado AND Nota >= Umbral
        
        y_true_dinamico = []
        for nota, estado in zip(notas_reales, estados_reales):
            es_riesgo = 0
            if "Reprobado" in estado: es_riesgo = 1
            elif nota < umbral: es_riesgo = 1 # Aprobó pero con nota baja (Riesgo)
            y_true_dinamico.append(es_riesgo)
            
        # Calcular Métricas
        # Comparamos la predicción FIJA del modelo (Reprueba/Aprueba) contra la realidad DINÁMICA
        prec = precision_score(y_true_dinamico, y_pred_model, zero_division=0)
        rec  = recall_score(y_true_dinamico, y_pred_model, zero_division=0)
        cm   = confusion_matrix(y_true_dinamico, y_pred_model)
        tn, fp, fn, tp = cm.ravel()
        
        # Guardar
        resultados_rdim.append({
            "RDIM": rdim,
            "Umbral_Riesgo": umbral,
            "Precision": prec,
            "Recall": rec,
            "Total_Riesgo": tp + fn, # Cuántos alumnos caen en esta categoría de riesgo
            "Predichos_Riesgo": tp + fp # Cuántos el modelo marcó como riesgo
        })
        
    return resultados_rdim

def main():
    vocab = build_vocab(DATA_DIR, reverse=True)
    
    # Definir umbrales: 4.0, 4.1, ..., 5.0
    umbrales = [round(x * 0.1, 1) for x in range(40, 51)]
    
    todos_resultados = []
    
    for rdim in RDIMS_SELECCIONADOS:
        res = analizar_umbral_riesgo(rdim, vocab, umbrales)
        todos_resultados.extend(res)
        
    if todos_resultados:
        df = pd.DataFrame(todos_resultados)
        print("\n================ RESULTADOS BARRIDO DE UMBRAL ================")
        # Mostrar tabla pivoteada para facilitar lectura
        pivot = df.pivot(index="Umbral_Riesgo", columns="RDIM", values=["Precision", "Recall"])
        print(pivot)
        
        out_csv = os.path.join(BASE_PATH, "barrido_umbrales_riesgo_reprobacion.csv")
        df.to_csv(out_csv, index=False)
        print(f"\n💾 Resultados guardados en: {out_csv}")
    else:
        print("⚠️ No se generaron resultados.")

if __name__ == "__main__":
    main()


⚡ Analizando RDIM=6...
   Evaluando 3023 registros contra umbrales variables...

⚡ Analizando RDIM=10...
   Evaluando 3023 registros contra umbrales variables...

⚡ Analizando RDIM=13...
   Evaluando 3023 registros contra umbrales variables...

⚡ Analizando RDIM=16...
   Evaluando 3023 registros contra umbrales variables...

================ RESULTADOS BARRIDO DE UMBRAL ================
              Precision                                  Recall            \
RDIM                 6         10        13        16        6         10   
Umbral_Riesgo                                                               
4.0            0.334928  0.334928  0.334928  0.334928  0.307018  0.307018   
4.1            0.368421  0.368421  0.368421  0.368421  0.289474  0.289474   
4.2            0.421053  0.421053  0.421053  0.421053  0.303448  0.303448   
4.3            0.459330  0.459330  0.459330  0.459330  0.296296  0.296296   
4.4            0.521531  0.521531  0.521531  0.521531  0.301105  0.301

# Modelo aumentado(dos cohortes)

### balanceo de datos

In [1]:
# -*- coding: utf-8 -*-
import os
import random

# ==========================================
# CONFIGURACIÓN
# ==========================================
# La carpeta donde tienes tu train.txt original
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales"

INPUT_FILE  = "train.txt"
OUTPUT_FILE = "train_balanceado.txt"

# Semilla para que el shuffle sea reproducible
random.seed(42)

def balancear_dataset():
    input_path = os.path.join(DATA_DIR, INPUT_FILE)
    output_path = os.path.join(DATA_DIR, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"❌ Error: No se encuentra el archivo {input_path}")
        return

    print(f"📂 Leyendo: {input_path}")
    
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    # 1. Separar clases
    # Asumimos que la relación de riesgo se llama "reprueba"
    # y cualquier otra cosa (aprueba, aprueba_4_5, etc.) es la clase mayoritaria
    reprobados = []
    aprobados = []

    for line in lines:
        # Dividimos por tabulación o espacio
        parts = line.split()
        if len(parts) < 3: continue
        
        # La relación suele ser el segundo elemento
        relacion = parts[1].lower()
        
        if "reprueba" in relacion:
            reprobados.append(line)
        else:
            aprobados.append(line)

    n_apr = len(aprobados)
    n_rep = len(reprobados)

    print(f"📊 Estadísticas Originales:")
    print(f"   - Aprobados (Mayoritaria): {n_apr}")
    print(f"   - Reprobados (Minoritaria): {n_rep}")

    if n_rep == 0:
        print("⚠️ No hay casos de reprobación. No se puede balancear.")
        return

    # 2. Calcular Factor de Balanceo
    # Queremos que Reprobados llegue a tener el mismo número que Aprobados
    if n_apr > n_rep:
        # Cuántas veces cabe la lista de reprobados en la de aprobados
        factor = n_apr // n_rep
        resto  = n_apr % n_rep
        
        print(f"⚖️  Balanceando... Se duplicarán los reprobados aprox {factor} veces.")
        
        # Multiplicamos la lista y agregamos lo que falte para igualar exacto
        reprobados_balanceados = (reprobados * factor) + reprobados[:resto]
    else:
        # Si ya están balanceados o reprobados son más (raro), lo dejamos igual
        reprobados_balanceados = reprobados

    # 3. Unir y Mezclar
    dataset_final = aprobados + reprobados_balanceados
    random.shuffle(dataset_final) # Importante: mezclar para que no queden ordenados

    print(f"📊 Estadísticas Finales:")
    print(f"   - Total Líneas: {len(dataset_final)}")
    print(f"   - (Aprobados: {len(aprobados)} vs Reprobados Aumentados: {len(reprobados_balanceados)})")

    # 4. Guardar
    with open(output_path, 'w', encoding='utf-8') as f:
        for line in dataset_final:
            f.write(line + "\n")

    print(f"✅ Archivo guardado exitosamente en: {output_path}")

if __name__ == "__main__":
    balancear_dataset()

📂 Leyendo: C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales\train.txt
📊 Estadísticas Originales:
   - Aprobados (Mayoritaria): 4584
   - Reprobados (Minoritaria): 201
⚖️  Balanceando... Se duplicarán los reprobados aprox 22 veces.
📊 Estadísticas Finales:
   - Total Líneas: 9168
   - (Aprobados: 4584 vs Reprobados Aumentados: 4584)
✅ Archivo guardado exitosamente en: C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales\train_balanceado.txt


### script entrenamiento

In [ ]:
:: ================================================================
:: ENTRENAMIENTO MULTIPLE DE MODELOS TUCKER (5D - NOTAS + PUNTAJE)
:: ================================================================

@echo off
ECHO ===============================================================
ECHO            INICIANDO ENTRENAMIENTOS MULTIDIMENSIONALES (5D)
ECHO ===============================================================

:: Configuración general
set DATASET=dataset_2019_2020_fundamentales
:: ⚠️ RUTAS ACTUALIZADAS A TUS NUEVOS ARCHIVOS 5D
set EMB_INIT=notebooks/Experimento_warm_start/embeddings_inicializados_normalizado_2019_2020.pt
set VOCAB_INIT=notebooks/Experimento_warm_start/vocabulario_normalizado_2019_2020.json

set BATCH=128
set LR=0.003
set PATIENCE=400
set EPOCHS=1000 

:: Lista de dimensiones de relación a entrenar
set RDIMS=1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16

:: Bucle principal
for %%R in (%RDIMS%) do (
    ECHO.
    ECHO ===============================================================
    ECHO Entrenando modelo con dimension de relaciones = %%R
    ECHO ===============================================================

    python main_original_warm_start_gemini_2_earlystopping_gemini.py ^
        --dataset %DATASET% ^
        --output_prefix Experimento_warm_start_rdim%%R_1000epochs_earlystopping_2019_2020_patience%PATIENCE%_balanceado_ ^
        --edim 4 ^
        --rdim %%R ^
        --num_iterations %EPOCHS% ^
        --batch_size %BATCH% ^
        --lr %LR% ^
        --init_embeddings %EMB_INIT% ^
        --init_vocab %VOCAB_INIT% ^
        --patience %PATIENCE%

    ECHO ---------------------------------------------------------------
    ECHO Modelo con rdim=%%R completado.
    ECHO ---------------------------------------------------------------
)

ECHO ===============================================================
ECHO TODOS LOS ENTRENAMIENTOS FINALIZADOS.
ECHO ===============================================================

pause

### Entrenar redes, balanceadas y no

In [4]:
#Balanceadas
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# -------------------------------------------------
# Rutas base (MODELO TUCKER 2019+2020)
# -------------------------------------------------
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales"

RESULTS_BASE    = r"C:\Users\56946\TuckER\results"
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience400_balanceado_"

BASE_DF_PATH    = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_modelobalanceado"

# cursos fundamentales
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PERMITIDOS = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = list(range(1, 17))  # 1..16

# =========================
# FUNCIONES AUXILIARES
# =========================
def cargar_df_notas(path_csv):
    """Lee un df de notas, normaliza ID y CURSO y devuelve el DataFrame."""
    df = pd.read_csv(path_csv, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    return df


def get_vocab_from_data_dir(data_dir):
    """Reconstruye vocabulario de entidades y relaciones desde train/valid/test."""
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt']:
        path = os.path.join(data_dir, part)
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                h, r, t = line.strip().split()
                entities.add(h); entities.add(t); relations.add(r)
    relations = sorted(list(relations))
    relations_with_reverse = relations + [r + '_reverse' for r in relations]
    return sorted(list(entities)), sorted(list(set(relations_with_reverse)))


def pick_state_dict(ckpt_loaded):
    if isinstance(ckpt_loaded, dict):
        if "model_state_dict" in ckpt_loaded:
            return ckpt_loaded["model_state_dict"]
        if "state_dict" in ckpt_loaded:
            return ckpt_loaded["state_dict"]
    return ckpt_loaded


def cargar_E_weights(path_pt):
    """Carga solo E.weight (embeddings de entidades) desde un checkpoint TuckER."""
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    if "E.weight" not in sd:
        raise KeyError(f"E.weight no está en el checkpoint: {path_pt}")
    return sd["E.weight"].detach().cpu()


def construir_notas_generacion(df_sem1, df_sem2,
                               cursos_primer, cursos_permitidos):
    """
    Para una generación (por ejemplo 2019 o 2020):

    1) Alumnos que toman los 4 cursos de primer semestre en sem1.
    2) De esos, nos quedamos con los que en sem2 inscriben >=1 curso en cursos_permitidos.
    3) Construimos vector de notas de sem1 (solo cursos_primer), normalizado /7.
    """
    # Alumnos con los 4 cursos fundamentales en sem1
    df_fund = df_sem1[df_sem1['CURSO'].isin(cursos_primer)]
    conteo = df_fund.groupby('ID')['CURSO'].nunique()
    alumnos_4 = conteo[conteo == len(cursos_primer)].index

    # De ellos, los que en sem2 toman alguno de los 8 fundamentales
    df_sem2_filt = df_sem2[
        (df_sem2['ID'].isin(alumnos_4)) &
        (df_sem2['CURSO'].isin(cursos_permitidos))
    ]
    alumnos_validos = sorted(df_sem2_filt['ID'].unique())

    if not alumnos_validos:
        return pd.DataFrame(columns=cursos_primer)

    # Pivot sem1 para esos alumnos
    notas = (
        df_sem1[
            (df_sem1['ID'].isin(alumnos_validos)) &
            (df_sem1['CURSO'].isin(cursos_primer))
        ]
        .pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
        .reindex(columns=cursos_primer)
        .fillna(0.0)
    )
    notas = notas / 7.0
    return notas


class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x):
        return self.network(x)


def entrenar_predictor(X_data, Y_data, save_path,
                       max_epochs=500, lr=1e-3,
                       patience=100, verbose_every=10):
    X_train_val, X_test, Y_train_val, Y_test = train_test_split(
        X_data, Y_data, test_size=0.20, random_state=42
    )
    X_train, X_val, Y_train, Y_val = train_test_split(
        X_train_val, Y_train_val, test_size=0.15, random_state=42
    )

    X_train_t = torch.FloatTensor(X_train)
    Y_train_t = torch.FloatTensor(Y_train)
    X_val_t   = torch.FloatTensor(X_val)
    Y_val_t   = torch.FloatTensor(Y_val)
    X_test_t  = torch.FloatTensor(X_test)
    Y_test_t  = torch.FloatTensor(Y_test)

    input_dim  = X_train_t.shape[1]
    output_dim = Y_train_t.shape[1]

    model = EmbeddingPredictor(input_dim, output_dim).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val = float('inf')
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_t), Y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            vloss = criterion(model(X_val_t), Y_val_t)

        if epoch % verbose_every == 0:
            print(f"  Epoch {epoch:03d}/{max_epochs} "
                  f"| train={loss.item():.6f} | val={vloss.item():.6f}")

        if vloss.item() < best_val - 1e-9:
            best_val = vloss.item()
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹️ Early stop en epoch {epoch} "
                      f"(sin mejora {patience} épocas).")
                break

    best = EmbeddingPredictor(input_dim, output_dim).to(DEVICE)
    best.load_state_dict(torch.load(save_path, map_location=DEVICE))
    best.eval()
    with torch.no_grad():
        test_mse = criterion(best(X_test_t), Y_test_t).item()
    return test_mse


# =========================
# PIPELINE PRINCIPAL
# =========================
def main():
    if not os.path.exists(SAVE_BASE):
        os.makedirs(SAVE_BASE)
        print(f"Created directory: {SAVE_BASE}")

    print("=== Entrenando red notas -> embeddings (modelo TuckER 2019+2020) ===")

    # ------------------ vocab TuckER combinado ------------------
    entities, relations = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    d = SimpleNamespace(
        entities      = entities,
        relations     = relations,
        entity_idxs   = {e: i for i, e in enumerate(entities)},
        relation_idxs = {r: i for i, r in enumerate(relations)}
    )

    # ------------------ cargar dataframes de notas ------------------
    df_20191 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20191.csv"))
    df_20192 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20192.csv"))
    df_20201 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20201.csv"))
    df_20202 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20202.csv"))

    # ------------------ construir X para cada generación ------------------
    notas_2019 = construir_notas_generacion(
        df_20191, df_20192,
        CURSOS_PRIMER, CURSOS_PERMITIDOS
    )
    notas_2020 = construir_notas_generacion(
        df_20201, df_20202,
        CURSOS_PRIMER, CURSOS_PERMITIDOS
    )

    # concatenamos ambas generaciones
    notas_por_alumno = pd.concat([notas_2019, notas_2020], axis=0)
    # por si acaso hubiera IDs duplicados, nos quedamos con la primera aparición
    notas_por_alumno = notas_por_alumno[~notas_por_alumno.index.duplicated(keep="first")]

    print(f"-> Vectores X normalizados preparados para {len(notas_por_alumno)} alumnos (2019+2020).")

    resumen = []
    print("\n=== Bucle por RDIM (TuckER 2019+2020) ===")
    for rdim in RDIMS:
        run_dir    = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim))
        tucker_pt = os.path.join(run_dir, "best_model.pt")
        save_path = os.path.join(
            SAVE_BASE, f"best_predictor_model_rdim{rdim}_normalizado_2019_2020.pt"
        )

        print(f"\n>> rdim={rdim} | checkpoint: {tucker_pt}")
        if not os.path.exists(tucker_pt):
            print("   ⚠️ No existe el checkpoint. Se omite.")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            EDIM = E.shape[1]

            entity_to_idx = d.entity_idxs
            alumnos_en_E = [e for e in entity_to_idx.keys()
                            if e.startswith("A") or e.isdigit()] # Assuming student IDs start with 'A' or are digits. Adjust if needed.
            # A more robust check might be needed if IDs have other formats.
            # Since the original script had this, I'll keep it but be aware.
            # Actually, based on previous functions, get_vocab_from_data_dir gets all entities.
            # And notas_por_alumno has the student IDs as index.
            # So we can just intersect directly with d.entity_idxs keys.
            alumnos_en_E = list(d.entity_idxs.keys())


            alumnos_comunes = sorted(
                set(notas_por_alumno.index).intersection(alumnos_en_E)
            )
            if not alumnos_comunes:
                print("   ⚠️ Sin alumnos comunes notas↔embeddings. Se omite.")
                continue

            X_data = np.vstack([notas_por_alumno.loc[a].values
                                for a in alumnos_comunes])
            idxs   = [entity_to_idx[a] for a in alumnos_comunes]
            Y_data = E[idxs].numpy()

            print(f"   -> EDIM={EDIM}, alumnos={len(alumnos_comunes)} (2019+2020)")
            test_mse = entrenar_predictor(
                X_data, Y_data, save_path,
                max_epochs=500, lr=1e-3, patience=100
            )
            print(f"   ✅ Guardado predictor en: {save_path}")
            print(f"   📏 MSE (test): {test_mse:.6f}")

            resumen.append((rdim, EDIM, len(alumnos_comunes), test_mse, save_path))

        except Exception as e:
            print(f"   ❌ Error en rdim={rdim}: {e}")
            continue

    if resumen:
        df_sum = pd.DataFrame(
            resumen,
            columns=["rdim","edim","n_alumnos","test_mse","ruta_modelo"]
        )
        out_csv = os.path.join(
            SAVE_BASE, "resumen_predictor_por_rdim_normalizado_2019_2020.csv"
        )
        df_sum.sort_values("rdim").to_csv(out_csv, index=False)
        print(f"\n=== Resumen guardado en: {out_csv} ===")
        print(df_sum.sort_values("rdim").to_string(index=False))
    else:
        print("\nNo se generaron resultados. Revisa rutas o datos.")


if __name__ == "__main__":
    main()

=== Entrenando red notas -> embeddings (modelo TuckER 2019+2020) ===
-> Vectores X normalizados preparados para 1571 alumnos (2019+2020).

=== Bucle por RDIM (TuckER 2019+2020) ===

>> rdim=1 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim1_1000epochs_earlystopping_2019_2020_patience400_balanceado_\best_model.pt
   -> EDIM=4, alumnos=1571 (2019+2020)
  Epoch 010/500 | train=0.425888 | val=0.376383
  Epoch 020/500 | train=0.072525 | val=0.030188
  Epoch 030/500 | train=0.080367 | val=0.027403
  Epoch 040/500 | train=0.047540 | val=0.013245
  Epoch 050/500 | train=0.045769 | val=0.016402
  Epoch 060/500 | train=0.041649 | val=0.007593
  Epoch 070/500 | train=0.037405 | val=0.007921
  Epoch 080/500 | train=0.034868 | val=0.009924
  Epoch 090/500 | train=0.033697 | val=0.007814
  Epoch 100/500 | train=0.032758 | val=0.009053
  Epoch 110/500 | train=0.030467 | val=0.008507
  Epoch 120/500 | train=0.028502 | val=0.009463
  Epoch 130/500 | train=0.027433 | val=0.009981

  Epoch 060/500 | train=0.043247 | val=0.009936
  Epoch 070/500 | train=0.039887 | val=0.011246
  Epoch 080/500 | train=0.036646 | val=0.012113
  Epoch 090/500 | train=0.036734 | val=0.010379
  Epoch 100/500 | train=0.033819 | val=0.011944
  Epoch 110/500 | train=0.033424 | val=0.011109
  Epoch 120/500 | train=0.031268 | val=0.011712
  Epoch 130/500 | train=0.031402 | val=0.011928
  Epoch 140/500 | train=0.030167 | val=0.011943
  Epoch 150/500 | train=0.029146 | val=0.012138
  Epoch 160/500 | train=0.028294 | val=0.012373
  ⏹️ Early stop en epoch 164 (sin mejora 100 épocas).
   ✅ Guardado predictor en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_modelobalanceado\best_predictor_model_rdim7_normalizado_2019_2020.pt
   📏 MSE (test): 0.009934

>> rdim=8 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim8_1000epochs_earlystopping_2019_2020_patience400_balanceado_\best_model.pt
   -> EDIM=4, alumnos=1571 (2019+2020)
  Epoch 010/500 | train=0.62

   -> EDIM=4, alumnos=1571 (2019+2020)
  Epoch 010/500 | train=0.547349 | val=0.499881
  Epoch 020/500 | train=0.110626 | val=0.065892
  Epoch 030/500 | train=0.089528 | val=0.035200
  Epoch 040/500 | train=0.056010 | val=0.024468
  Epoch 050/500 | train=0.055352 | val=0.023972
  Epoch 060/500 | train=0.047745 | val=0.013541
  Epoch 070/500 | train=0.045614 | val=0.016212
  Epoch 080/500 | train=0.043150 | val=0.014105
  Epoch 090/500 | train=0.041206 | val=0.015303
  Epoch 100/500 | train=0.038418 | val=0.014013
  Epoch 110/500 | train=0.035619 | val=0.015616
  Epoch 120/500 | train=0.034534 | val=0.015679
  Epoch 130/500 | train=0.032708 | val=0.014889
  Epoch 140/500 | train=0.033866 | val=0.015207
  Epoch 150/500 | train=0.031675 | val=0.015661
  Epoch 160/500 | train=0.030280 | val=0.015557
  ⏹️ Early stop en epoch 163 (sin mejora 100 épocas).
   ✅ Guardado predictor en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_modelobalanceado\best_predictor_model_rd

### probar modelo

In [6]:
# -*- coding: utf-8 -*-
import os, re, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from types import SimpleNamespace
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# =====================================
# CONFIGURACIÓN DE RUTAS (MODELO BALANCEADO)
# =====================================

# ⚠️ CORRECCIÓN: Agregué la barra final '/' para que no concatene mal el nombre del archivo
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_fundamentales/"

# 2. Rutas de Datos de Evaluación (2021)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20131 = os.path.join(BASE_PATH, "df_20211.csv")  # Input (S1)
CSV_20132 = os.path.join(BASE_PATH, "df_20212.csv")  # Target (S2)

# 3. Rutas de Modelos (TuckER y Red Neuronal)
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# Prefijo del TuckER Balanceado
RUN_PREFIX   = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_patience300"

# Carpeta donde guardaste las redes neuronales balanceadas
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_modelobalanceado"

# Nombre del archivo de la red neuronal
PREDICTOR_FILE_FMT = "best_predictor_model_rdim{rdim}_normalizado_2019_2020.pt"

# Cursos
CURSOS_PRIMER  = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO = ['CC1002','MA1002','MA1102','FI1100']
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

DEVICE = "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)

RDIMS = list(range(1, 17))

# =========================
# UTILIDADES
# =========================
sys.path.append("C:/Users/56946/TuckER")
from load_data import Data

def build_vocab(data_dir, reverse=True):
    # Data espera que data_dir termine en / para concatenar "train.txt"
    d = Data(data_dir=data_dir, reverse=reverse)
    ent2idx = {e:i for i,e in enumerate(d.entities)}
    rel2idx = {r:i for i,r in enumerate(d.relations)}
    return SimpleNamespace(entities=d.entities, relations=d.relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_tucker_weights(path, device="cpu"):
    sd = pick_state_dict(torch.load(path, map_location=device))
    E = sd["E.weight"].to(device)
    R = sd["R.weight"].to(device)
    W = sd["W"].to(device)
    return E, R, W

def pick_forward_rel_idx(vocab, name):
    if name in vocab.relation_idxs: return vocab.relation_idxs[name]
    cands = [r for r in vocab.relations if r.replace("_reverse","") == name]
    if not cands: cands = [r for r in vocab.relations if re.search(name, r, re.I)]
    if not cands: raise KeyError(f"No encontré relación '{name}' en el vocab.")
    cands.sort(key=lambda r: ("_reverse" in r, r))
    return vocab.relation_idxs[cands[0]]

def contract_M(W, r_vec, d_e):
    if W.shape[0] == r_vec.numel() and W.shape[1] == d_e:
        return torch.tensordot(W, r_vec, dims=([0],[0]))
    if W.shape[1] == r_vec.numel() and W.shape[0] == d_e:
        return torch.tensordot(W, r_vec, dims=([1],[0]))
    raise ValueError(f"Layout W no reconocido: {tuple(W.shape)}")

# ===== Predictor =====
class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def load_predictor(path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=device)
    model.load_state_dict(pick_state_dict(state), strict=True)
    model.to(device).eval()
    return model

def notas_vector(csv_path, alumno_id, cursos_primer):
    df = pd.read_csv(csv_path, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    sub = df[(df['ID']==alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    if not sub.empty and sub['NOTA'].isna().all(): sub['NOTA'] = 0.0
    if sub.empty: return torch.zeros((1,len(cursos_primer)), dtype=torch.float32)
    piv = (sub.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
              .reindex(columns=cursos_primer).fillna(0.0))
    vec = piv.iloc[0].values.astype('float32') / 7.0
    return torch.tensor(vec).view(1,-1)

# =========================
# EVALUACIÓN
# =========================
def evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab):
    print(f"\n=============== Evaluando modelo: {tag} (BALANCEADO) ===============")
    
    try:
        E, R, W = load_tucker_weights(tucker_ckpt, device=DEVICE)
        d_e = E.shape[1]
        
        # Índices de relaciones
        ridx_apr  = pick_forward_rel_idx(vocab, "aprueba")
        try:
            ridx_repr = pick_forward_rel_idx(vocab, "reprueba")
            have_repr = True
        except KeyError:
            have_repr = False
            print("⚠️ No hay relación 'reprueba'. Usando 1 - P(aprueba).")

        predictor = load_predictor(predictor_ckpt, input_size=len(CURSOS_PRIMER), out_dim=d_e, device=DEVICE)
    except Exception as e:
        print(f"❌ Error cargando modelo {tag}: {e}")
        return None

    # Datos
    df_31 = pd.read_csv(CSV_20131, sep=';')
    df_32 = pd.read_csv(CSV_20132, sep=';')
    for df in (df_31, df_32):
        df['ID']    = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    ok = (df_31.groupby("ID")["CURSO"].apply(set)
             .apply(lambda s: set(CURSOS_PRIMER).issubset(s)))
    alumnos_validos = ok[ok].index.tolist()

    df_eval = df_32[(df_32['ID'].isin(alumnos_validos)) &
                    (df_32['CURSO'].isin(CURSOS_EVAL))].copy()

    df_eval['APROB'] = ((~df_eval['NOTA'].isna()) & (df_eval['NOTA'] >= 4.0)).astype(int)
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())].copy()
    
    if df_eval.empty:
        print("⚠️ Sin datos para evaluar.")
        return None

    ehat_cache = {}
    rows = []
    y_true_cls, y_pred_cls = [], []

    with torch.no_grad():
        M_apr = contract_M(W, R[ridx_apr], d_e)
        M_repr = contract_M(W, R[ridx_repr], d_e) if have_repr else None

        for _, r in df_eval.iterrows():
            aid, curso = r['ID'], r['CURSO']
            if aid not in ehat_cache:
                x = notas_vector(CSV_20131, aid, CURSOS_PRIMER).to(DEVICE)
                ehat_cache[aid] = predictor(x).squeeze(0).cpu()
            e_h = ehat_cache[aid]
            e_t = E[vocab.entity_idxs[curso]].cpu()

            s_apr = torch.sigmoid((e_h.view(1,d_e) @ M_apr @ e_t.view(d_e,1)).squeeze()).item()
            if have_repr:
                s_repr = torch.sigmoid((e_h.view(1,d_e) @ M_repr @ e_t.view(d_e,1)).squeeze()).item()
            else:
                s_repr = 1.0 - s_apr

            # Clasificación (1 = Reprueba)
            es_reprobado_real = 1 if r['APROB'] == 0 else 0
            predice_reprobado = 1 if s_repr > s_apr else 0
            
            y_true_cls.append(es_reprobado_real)
            y_pred_cls.append(predice_reprobado)

            # Ranking
            if r['APROB'] == 1:
                rank = 1 if s_apr >= s_repr else 2
            else:
                rank = 1 if s_repr >= s_apr else 2

            rows.append({
                "ID": aid, "CURSO": curso, 
                "hit@1": 1.0 if rank == 1 else 0.0,
                "hit@2": 1.0
            })

    df_out = pd.DataFrame(rows)
    hits1 = df_out["hit@1"].mean()
    mrr   = (1.0 / (2.0 - df_out["hit@1"])).mean() # Simplificado para binario (rank 1 o 2)
    N     = len(df_out)

    prec_repro = precision_score(y_true_cls, y_pred_cls, zero_division=0)
    rec_repro  = recall_score(y_true_cls, y_pred_cls, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true_cls, y_pred_cls).ravel()

    print("\n=== MÉTRICAS CLASE REPROBADOS ===")
    print(f"Precision : {prec_repro:.4f}")
    print(f"Recall    : {rec_repro:.4f}")
    print(f"Matriz    : [TN={tn} (Ok), FP={fp} (Falsa Alarma)]")
    print(f"            [FN={fn} (No Detectado), TP={tp} (Detectado)]")
    print(f"Hits@1    : {hits1:.4f}")

    return {
        "tag": tag,
        "Hits@1": hits1,
        "MRR": mrr,
        "Precision_Rep": prec_repro,
        "Recall_Rep": rec_repro,
        "N": N
    }

def main():
    vocab = build_vocab(DATA_DIR, reverse=True)
    res_list = []
    
    print(f"📂 Evaluando modelos en: {PREDICTOR_BASE}")
    
    for rdim in RDIMS:
        tag = f"rdim{rdim}"
        tucker_ckpt = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim), "best_model.pt")
        predictor_ckpt = os.path.join(PREDICTOR_BASE, PREDICTOR_FILE_FMT.format(rdim=rdim))

        if not os.path.exists(tucker_ckpt):
            print(f"⏩ Faltan archivos TuckER rdim={rdim}")
            continue
        if not os.path.exists(predictor_ckpt):
            print(f"⏩ Faltan archivos Red Neuronal rdim={rdim}")
            continue

        res = evaluar_modelo(tag, tucker_ckpt, predictor_ckpt, vocab)
        if res: res_list.append(res)

    if res_list:
        df_sum = pd.DataFrame(res_list).sort_values("tag")
        out_sum = os.path.join(PREDICTOR_BASE, "resumen_evaluacion_balanceado.csv")
        df_sum.to_csv(out_sum, index=False)
        print("\n================ RESUMEN MODELOS BALANCEADOS ================")
        print(df_sum.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        print(f"\n💾 Guardado en: {out_sum}")

if __name__ == "__main__":
    main()

📂 Evaluando modelos en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\redes_2cohortes_modelobalanceado

=============== Evaluando modelo: rdim1 (BALANCEADO) ===============

=== MÉTRICAS CLASE REPROBADOS ===
Precision : 0.0614
Recall    : 0.7468
Matriz    : [TN=132 (Ok), FP=2658 (Falsa Alarma)]
            [FN=59 (No Detectado), TP=174 (Detectado)]
Hits@1    : 0.1012

=============== Evaluando modelo: rdim2 (BALANCEADO) ===============

=== MÉTRICAS CLASE REPROBADOS ===
Precision : 0.3349
Recall    : 0.3004
Matriz    : [TN=2651 (Ok), FP=139 (Falsa Alarma)]
            [FN=163 (No Detectado), TP=70 (Detectado)]
Hits@1    : 0.9001

=============== Evaluando modelo: rdim3 (BALANCEADO) ===============

=== MÉTRICAS CLASE REPROBADOS ===
Precision : 0.0659
Recall    : 0.6052
Matriz    : [TN=792 (Ok), FP=1998 (Falsa Alarma)]
            [FN=92 (No Detectado), TP=141 (Detectado)]
Hits@1    : 0.3086

=============== Evaluando modelo: rdim4 (BALANCEADO) ===============

=== MÉTRICAS CLA

### Modelo 5d, aumentado y no